In [ ]:
# 第9周-Day5：DigitalEmployeeDefinition — 为什么数字员工不拥有 Runtime？
# matplotlib 中文字体配置
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## 📅 Week 9 - Day 5 | 2026-07-31

### DigitalEmployeeDefinition — 为什么数字员工不拥有 Runtime？

| 项目 | 内容 |
|---|---|
| **周主题** | Domain Deep Dive — 拆对象，理解为什么存在、边界在哪 |
| **今日对象** | DigitalEmployeeDefinition (BD-01) |
| **核心问题** | 为什么数字员工不拥有 Runtime？ |
| **ADR 依据** | ADR-006 §4.1 D-1, Domain Model §6.1 BD-01, §10.4-12 |
| **代码文件** | `digital_employee_model.py`

## ❓ 今日核心问题

**为什么 DigitalEmployeeDefinition 不拥有 Runtime？**

换个问法：如果数字员工是"员工"，为什么它不能"自己跑"？

在当前代码里，`DigitalEmployeeModel` 有 `bound_skill_id`、有 `kill_switch`、有 `status=active`——看起来它就是在"运行"。但 ADR-006 和 Domain Model §6.1 明确规定：**DigitalEmployeeDefinition 不拥有 Runtime，不持有 Deployment 状态，不构造 FrozenExecutionContext，不充当执行入口**。

## 🗣 人话解释（Jason 26年 ERP 经验）

在传统 ERP 里，"员工"是一个主数据对象——张三是采购员，李四是审批人。系统不会因为"张三"存在就自动运行采购流程。

LangChat 的数字员工也一样。**"数字员工"是身份，不是引擎。**

- 小明的**定义**（DigitalEmployeeDefinition）：名字、职责、关联的契约 → 像 HR 档案
- 小明的**能力**（SkillRelease）：具体执行的技能包 → 像岗位技能认证
- 小明的**部署**（Deployment/Revision）：在某环境实际运行 → 像被派到具体项目
- 小明的**运行时**（Runtime + FrozenExecutionContext）：每次执行 → 像每次具体干活

**DigitalEmployeeDefinition 就是 HR 档案。它只持有引用，不持有内容字节，不持有运行状态。**

In [ ]:
# 四层架构中 DigitalEmployeeDefinition 的位置
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)

layers = [
    (8, 'Business Domain Layer', '#E8F5E9', [
        'DigitalEmployeeDefinition（语义锚点）',
        '  ├── 引用 ApplicationContractVersion（业务契约）',
        '  ├── 引用 BlueprintVersion（谱系锚点）',
        '  └── 声明发布策略 scope',
        'ApplicationContract / ContractVersion (BD-02/03)',
        'Capability (SC-07)  KnowledgeCollection (SC-09)  Policy (SC-11)',
    ]),
    (6, 'Supply Chain Layer', '#FFF3E0', [
        'Blueprint → Build → ExecutionPlanIR → SkillRelease（制品链）',
        'ReleaseChannel (SC-14) → PromotionEvent (SC-15)',
        'CapabilityRelease / KnowledgeSnapshot / PolicyBundle（不可变版本）',
    ]),
    (4, 'Runtime Layer', '#E3F2FD', [
        'Deployment → DeploymentRevision → FrozenExecutionContext → Execution',
        'TrafficPolicy (RT-03)  Session / State / Memory',
    ]),
    (2, 'Operations Layer', '#F3E5F5', [
        'Registry / Catalog Projection / Governance',
        'Attestation / Provenance / Signature',
    ]),
]

colors = {'y': []}
for y, name, color, items in layers:
    rect = plt.Rectangle((0.5, y - 0.8), 13, 1.8, fill=True, facecolor=color, edgecolor='#333', linewidth=1.5, alpha=0.8)
    ax.add_patch(rect)
    ax.text(7, y + 0.7, name, ha='center', va='center', fontsize=13, fontweight='bold', color='#333')
    for j, item in enumerate(items):
        ax.text(1.2, y + 0.3 - j * 0.3, item, ha='left', va='center', fontsize=8.5, color='#555')

# 突出 DED
rect_ded = plt.Rectangle((0.5, 7.2), 13, 1.8, fill=False, edgecolor='#E53935', linewidth=3)
ax.add_patch(rect_ded)
ax.annotate('DED 在最上层（Business Domain）\nRuntime 在第三层\n跨越两层 — 定义层不触碰运行层',
            xy=(7, 7.2), xytext=(7, 0.3), ha='center', fontsize=9, color='#E53935',
            arrowprops=dict(arrowstyle='->', color='#E53935', lw=2))

ax.set_title('DigitalEmployeeDefinition 在四层架构中的位置', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 📋 ADR 依据

### ADR-006 D-1（§4.1）：DigitalEmployeeDefinition 是引用语义锚点，不是聚合根

**身份**：`(tenant, workspace, digital_employee_id, definition_version)`

**引用而非持有**：
- MUST 引用一份 ApplicationContractVersion digest
- MUST 引用一份活跃 BlueprintVersion digest 作为谱系锚点
- MUST 显式声明发布策略 scope
- MUST NOT 持有 Blueprint 内容字节、Knowledge 内容字节、Policy 内容字节、Deployment 状态或 Runtime 对象

**生命周期**：`Draft → Published → Deprecated → Retired`（没有 Activated 状态！）

### ADR-006 §7 明确不做（前 7 条）
1. 不拥有 Runtime
2. 不持有 Deployment 状态
3. 不持有任何 artifact 内容
4. 不签发授权
5. 不构造 FrozenExecutionContext
6. 不充当执行入口
7. 不聚合子内容为巨型聚合根

In [ ]:
# 当前代码 vs 目标态 Gap 分析
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')

headers = ['目标态要求', '当前代码', 'Gap']
data = [
    ['definition_version 单调递增', '无版本字段', '❌ 缺失'],
    ['引用 ApplicationContractVersion digest', '不存在', '❌ 缺失'],
    ['引用 BlueprintVersion digest', '不存在', '❌ 缺失'],
    ['声明发布策略 scope', '不存在', '❌ 缺失'],
    ['生命周期 Draft→Published→Deprecated→Retired', 'active/inactive/retired', '⚠️ 三态vs四态'],
    ['只持有引用不持有内容', 'bound_skill_id 是 tag 引用', '⚠️ 用 skill_id'],
    ['不拥有 Runtime', 'kill_switch 直接控制执行', '⚠️ 跨层'],
    ['不签发授权', 'allowed_capabilities_json', '⚠️ 声明与授权混合'],
]

# Draw table
col_widths = [4.5, 4, 3]
col_starts = [0.5, 0.5 + col_widths[0] + 0.2, 0.5 + col_widths[0] + 0.2 + col_widths[1] + 0.2]
row_height = 0.65

# Header
for i, h in enumerate(headers):
    rect = plt.Rectangle((col_starts[i], 8), col_widths[i], row_height, facecolor='#37474F', edgecolor='#263238')
    ax.add_patch(rect)
    ax.text(col_starts[i] + col_widths[i]/2, 8 + row_height/2, h, ha='center', va='center', fontsize=11, fontweight='bold', color='white')

# Rows
gap_colors = {'❌ 缺失': '#FFCDD2', '⚠️ 三态vs四态': '#FFF9C4', '⚠️ 用 skill_id': '#FFF9C4', '⚠️ 跨层': '#FFF9C4', '⚠️ 声明与授权混合': '#FFF9C4'}
for r, row in enumerate(data):
    y = 8 - (r + 1) * row_height
    for c, val in enumerate(row):
    bg = gap_colors.get(val, '#E8F5E9') if c == 2 else ('#F5F5F5' if r % 2 == 0 else '#FFFFFF')
    rect = plt.Rectangle((col_starts[c], y), col_widths[c], row_height, facecolor=bg, edgecolor='#E0E0E0')
    ax.add_patch(rect)
    ax.text(col_starts[c] + 0.15, y + row_height/2, val, ha='left', va='center', fontsize=9)

ax.text(7, 9.2, 'DigitalEmployeeModel 当前代码 vs 目标态 Gap', ha='center', fontsize=14, fontweight='bold')
ax.set_xlim(0, 14)
ax.set_ylim(1.5, 9.8)
plt.tight_layout()
plt.show()

print("\n关键发现：status=active 同时承载了'定义已发布'和'部署在服役'双重语义")
print("目标态要拆分为 DigitalEmployeeDefinition.Published + Deployment.Active 两个独立状态")

## 🏢 商业地产映射

| LangChat 概念 | MI CRE 场景 | 说明 |
|---|---|---|
| DigitalEmployeeDefinition | "合同审核数字员工" HR 档案 | 定义身份、职责、归属部门 |
| ApplicationContractVersion | 合同审核岗位 SOP | 输入输出、权限要求 |
| BlueprintVersion | 审核流程设计文档 | 经评审的规范文件 |
| SkillRelease | 审核技能包 | 含模型+知识+策略 |
| Deployment | 派驻到某 mall 的审核岗 | 在某环境实际部署 |
| FrozenExecutionContext | 每次审核任务的工作令 | 身份、权限、知识版本全部冻结 |

**MI 场景**：定义一个"合同审核数字员工"叫小合 → 部署到上海 mall 和北京 mall → 两个独立 Deployment → 北京暂停不影响上海。

In [ ]:
# 定义拥有 Runtime vs 定义不拥有 Runtime
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

dimensions = ['多环境部署', '灰度发布', '回滚', '审计', '知识更新', '权限分离']
plan_a = [1, 0, 1, 1, 1, 1]  # 方案A（传统）评分 0-2
plan_b = [2, 2, 2, 2, 2, 2]  # 方案B（LangChat）

import numpy as np
x = np.arange(len(dimensions))
width = 0.35

bars_a = ax.bar(x - width/2, plan_a, width, label='方案A：定义拥有 Runtime（传统）', color='#EF9A9A', edgecolor='#E53935')
bars_b = ax.bar(x + width/2, plan_b, width, label='方案B：定义不拥有 Runtime（LangChat v2）', color='#A5D6A7', edgecolor='#43A047')

ax.set_xticks(x)
ax.set_xticklabels(dimensions, fontsize=10)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['❌ 不支持', '⚠️ 有限', '✅ 原生支持'], fontsize=9)
ax.set_ylim(0, 2.5)
ax.legend(fontsize=10, loc='upper left')
ax.set_title('数字员工应该拥有 Runtime 吗？', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 🧠 架构师思考题

**场景**：MI 集团有 10 个 mall，每个 mall 都需要"合同审核数字员工"。这 10 个数字员工：
- 使用相同的合同审核 SOP
- 但每个 mall 的法规知识库不同
- 每个 mall 的策略不同

**问题**：
1. 这需要几个 DigitalEmployeeDefinition？几个 Deployment？
2. 如果上海 mall 的知识库更新了，会影响北京 mall 吗？
3. kill_switch 应该属于定义层还是 Deployment 层？

## 💡 我的理解变化

**以前以为**：数字员工就是一个"智能体"——定义它、启动它、它就开始干活。

**现在知道**：数字员工是一组架构对象的协作——
- DigitalEmployeeDefinition 是**身份声明**（HR 档案）
- SkillRelease 是**能力制品**（技能认证）
- Deployment/Revision 是**部署实例**（外派到项目）
- FrozenExecutionContext 是**每次工作的许可令**
- Execution 是**一次具体工作**

**最反直觉**：当前代码的 `kill_switch` 直接放在 DigitalEmployeeModel 上，看起来合理。但目标态下 kill_switch 应该在 Deployment 层——"停止运行"是运行时决策，不是定义层决策。

## 🔗 明日连接

**Day6（周六）：⚡ 画 Domain Model Diagram** — 把 Week 9 所有对象的关系图画出来。

## 📖 术语表

| 英文术语 | 音标 | 释义 |
|---|---|---|
| DigitalEmployeeDefinition | /ˈdɪdʒɪtl ˈɛmplɔɪiː ˌdɛfɪˈnɪʃən/ | 数字员工定义（语义锚点） |
| FrozenExecutionContext | /ˈfroʊzən ɪkˈstɛkst/ | 不可变执行上下文 |
| Deployment Revision | /dɪˈplɔɪmənt ˈriːvɪʒən/ | 部署修订版本（运行时闭包） |
| Semantic Anchor | /sɪˈmæntɪk ˈæŋkər/ | 语义锚点（引用中心） |
| Kill Switch | /kɪl swɪtʃ/ | 紧急停止开关 |